# WebSwipe decoder: training + evaluation on a free GPU (Colab or Kaggle)

**Prepare (on the laptop, takes seconds):** `bash tools/make_colab_bundle.sh` -> `colab_bundle.zip` (~115 MB:
code, D1 features, raw D1 traces).

**Colab:** upload `colab_bundle.zip` to `MyDrive/webswipe/`. Runtime > Change runtime type > T4 GPU. Run all.
Checkpoints and results go to `MyDrive/webswipe/` (survive disconnects).

**Kaggle:** Datasets > New dataset > upload `colab_bundle.zip` (Kaggle unpacks it; either form works).
New notebook > File > Import notebook (this file) > Add input: your dataset. Settings: Accelerator = GPU,
Internet = on (for `pip install wordfreq`). Run all, or *Save & Run All* to run in the background (up to 12 h).
Results go to `/kaggle/working/out` (Output tab).

Every training command prints seconds per epoch: check the first epoch and adjust `EPOCHS` if needed.

In [ ]:
import glob, os, subprocess

ON_KAGGLE = os.path.exists('/kaggle/input')
if ON_KAGGLE:
    ROOT, WS = '/kaggle/working/out', '/kaggle/working/ws'
    zips = glob.glob('/kaggle/input/**/colab_bundle.zip', recursive=True)
else:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT, WS = '/content/drive/MyDrive/webswipe', '/content/ws'
    zips = [f'{ROOT}/colab_bundle.zip']

subprocess.run(['rm', '-rf', WS], check=True)
os.makedirs(f'{WS}/data', exist_ok=True)
os.makedirs(f'{ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{ROOT}/results', exist_ok=True)
if zips and os.path.exists(zips[0]):
    subprocess.run(['unzip', '-q', zips[0], '-d', WS], check=True)
else:  # Kaggle unpacked the zip into a folder
    pkg = glob.glob('/kaggle/input/**/webswipe/__init__.py', recursive=True)
    assert pkg, 'bundle not found: add the dataset with colab_bundle.zip as an input'
    subprocess.run(f'cp -r {os.path.dirname(os.path.dirname(pkg[0]))}/* {WS}/', shell=True, check=True)
subprocess.run(f'mv {WS}/*.npz {WS}/shen_midair.pkl {WS}/data/', shell=True, check=True)
os.chdir(WS)
print('platform:', 'Kaggle' if ON_KAGGLE else 'Colab', '| outputs ->', ROOT)
print(sorted(os.listdir('data')))

In [ ]:
!pip -q install wordfreq
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 0. Smoke test (~1 min): one tiny epoch + tiny evaluations, to catch setup problems early

In [ ]:
!python -u -m webswipe.decoder.train --split subject --test-subject 0:0 --epochs 1 --max-train 256 --aug-prob 0.5 --out /tmp/smoke.pt
!python -m webswipe.eval.robustness_d1 --resample-hz 50 --max-test 20 --ckpt smoke=/tmp/smoke_last.pt --out /tmp/smoke_rob.json
!python -m webswipe.eval.lm_prior_d1 --resample-hz 50 --max-test 20 --n-tune 50 --ckpt smoke=/tmp/smoke_last.pt --out /tmp/smoke_lm.json

## 1. Main experiment: does webcam augmentation help? (same budget, held-out subject 0)
`clean` = trained as in Shen et al.; `aug` = half of the training gestures are re-featurised from a
webcam-like degraded trace each epoch (random 10-30 fps, jitter up to 0.3 keys, pinch timing up to 100 ms).
Reported numbers use the **last** epoch (`*_last.pt`); the "best" checkpoint is chosen on the test subject.

In [ ]:
EPOCHS = 100
!python -u -m webswipe.decoder.train --split subject --test-subject 0:0 --epochs {EPOCHS} --eval-every 10 --out {ROOT}/checkpoints/d1_loso0_clean.pt

In [ ]:
!python -u -m webswipe.decoder.train --split subject --test-subject 0:0 --epochs {EPOCHS} --eval-every 10 --aug-prob 0.5 --out {ROOT}/checkpoints/d1_loso0_aug.pt

## 2. Evaluation: SHARK2 vs both models on identical degraded traces, then the word-frequency prior
CPU-bound (~10-20 min each). Results: `results/robustness_d1.json`, `results/lm_prior_d1.json`,
`results/robustness_d1.png` -> copy them into the repo's `results/` and fill in `results/README.md`.

In [ ]:
CK = f'clean={ROOT}/checkpoints/d1_loso0_clean_last.pt aug={ROOT}/checkpoints/d1_loso0_aug_last.pt'
!python -m webswipe.eval.robustness_d1 --resample-hz 50 --ckpt {CK} --out {ROOT}/results/robustness_d1.json
!python -m webswipe.eval.lm_prior_d1 --resample-hz 50 --ckpt {CK} --out {ROOT}/results/lm_prior_d1.json
!python -m webswipe.eval.plot_robustness --json {ROOT}/results/robustness_d1.json --out {ROOT}/results/robustness_d1.png

## 3. Optional, for the final report
**Reproduction** of Shen et al. (random 95/5 split, 200 epochs, as in their code):

In [ ]:
# !python -u -m webswipe.decoder.train --epochs 200 --eval-every 10 --out {ROOT}/checkpoints/shen_d1_random.pt

**All 9 held-out subjects** (D1 subject ids: 0 1 2 4 5 6 7 8 9). Long: run on Kaggle with *Save & Run All*,
or split the list across sessions. Re-run the evaluations per subject with `--test-subject <id>`.

In [ ]:
# for s in [1, 2, 4, 5, 6, 7, 8, 9]:
#     for name, p in [('clean', 0.0), ('aug', 0.5)]:
#         !python -u -m webswipe.decoder.train --split subject --test-subject 0:{s} --epochs {EPOCHS} --eval-every 50 --aug-prob {p} --out {ROOT}/checkpoints/d1_loso{s}_{name}.pt

## 4. Fine-tune on our webcam data (after recording D3)
Laptop: `python -m webswipe.decoder.webcam_data --resample-hz 50` -> `data/webcam_features.npz`, re-run
`tools/make_colab_bundle.sh` (it picks up every `data/*_features.npz`), re-upload.
Subject ids are `<file index>:<participant>`; with `--data data/webcam_features.npz` alone the index is 0.

In [ ]:
# !python -u -m webswipe.decoder.train --data data/webcam_features.npz --split subject --test-subject 0:<participant> \
#     --init {ROOT}/checkpoints/d1_loso0_aug_last.pt --lr 3e-5 --epochs 50 --eval-every 5 --out {ROOT}/checkpoints/ft_webcam.pt